# MothRAG quickstart — custom corpus from text files

This notebook walks through the 5-minute path from raw documents to question-answering.

## What you'll do

1. Load a small mixed corpus (`.md` + `.txt` + `.json`) with one line.
2. Verify the auto-default backends MothRAG picked.
3. Ask a few questions and inspect the routing decisions.
4. (Optional) Plug in a production reader if you have an API key.

**No API keys required.** Out of the box MothRAG falls back to deterministic offline backends so this notebook runs end-to-end on any laptop.

## 1. Install

In [ ]:
# Run once (or skip if you already installed mothrag).
# %pip install -q mothrag

## 2. Load a small mixed corpus

The `examples/sample_corpus/` folder ships with three documents in different formats. `MothRAG.from_documents` recursively walks the folder, dispatches the right loader per extension, chunks, embeds, and indexes — all in one call.

In [ ]:
from mothrag import MothRAG

rag = MothRAG.from_documents("sample_corpus/")
rag

Notice the `repr` shows you which backends were auto-selected. In a clean install with no API keys, you'll see:

- `embedder=SentenceTransformersEmbedder` (the MiniLM-L6 fallback) **or** `_HashEmbedder` (if `sentence-transformers` isn't installed)
- `reader=_EchoReader` (the offline answer extractor)
- `vector_db=_MemoryVectorStore` (in-memory cosine-similarity index)
- `indexed=N` chunks created from your documents

## 3. Ask questions

Each call to `query` returns a `QueryResult` with the answer **and** routing provenance — useful for citations and debugging.

In [ ]:
result = rag.query("Who created Python?")
print("Answer:    ", result.answer)
print("Arm subset:", result.arm_subset)
print("Retrieved: ", len(result.retrieved_chunks), "chunks")
result.retrieved_chunks[0].metadata

In [ ]:
result = rag.query("How tall is the Eiffel Tower?")
print("Answer:", result.answer)

In [ ]:
result = rag.query("Which Christopher Nolan film was released first, Memento or Inception?")
print("Answer:    ", result.answer)
print("Arm subset:", result.arm_subset, "<-- comparative-selection signal expected")

## 4. Parallel batch

When you have many questions, `batch_query` runs them in a thread pool.

In [ ]:
questions = [
    "Who created Python?",
    "When was the Eiffel Tower built?",
    "Who directed Inception?",
]
for q, r in zip(questions, rag.batch_query(questions, max_workers=3)):
    print(f"Q: {q}")
    print(f"A: {r.answer}\n")

## 5. Upgrade to a production reader (optional)

Set one environment variable (or `dotenv`) before constructing `MothRAG` and the production reader is picked up automatically:

```python
import os
os.environ["GROQ_API_KEY"]     = "..."      # Llama-3.3-70B via Groq (the paper's reader)
# or
os.environ["TOGETHER_API_KEY"] = "..."      # Llama-3.3-70B via Together

rag = MothRAG.from_documents("sample_corpus/")
print(rag.reader)   # → OpenAICompatibleReader(...)
```

Or pass an explicit reader via the constructor (works without env vars):

```python
from mothrag.core._api_adapters import OpenAICompatibleReader

reader = OpenAICompatibleReader(
    model="llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key="<GROQ_API_KEY>",
)
rag = MothRAG.from_documents("sample_corpus/", reader=reader)
```

## Next

- **Paper-grade reproduction** (Llama via Groq + Gemini Embedding 2 + γ verifier): `scripts/route_prospective.py`. See the [Production deployment guide](../docs/guide/production.md).
- **Plug-in custom backends**: implement the `Embedder` / `Reader` / `VectorStore` protocols. See the [API reference](../docs/guide/api.md).